In [22]:
import torch
import torch.nn as nn

import pandas as pd

from src.preprocessing import build_preprocessor
from src.preprocessing import AgeImputer, EmbarkedImputer, HasCabinTransformer

import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline

In [ ]:
#%cd ..

%load_ext autoreload
%autoreload 2

c:\Users\coffe\Desktop\titanic-ml\titanic-ml


In [4]:
SEED = 42

In [5]:
class TitanicMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32, bias=False),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            
            nn.Linear(32, 32, bias=False),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            
            nn.Linear(32, 1)
        )
        
    def forward(self, x):
        return self.net(x)

In [6]:
from sklearn.model_selection import train_test_split

train = pd.read_csv("data/train.csv")

X = train.drop(columns=['Survived'])
y = train['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)

In [7]:
preprocessor = build_preprocessor('standard')

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_test)

X_train_processed.shape, X_val_processed.shape

((712, 13), (179, 13))

In [8]:
X_train_tensor = torch.tensor(X_train_processed, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_processed, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
y_val_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

print(X_train_tensor.shape)
print(y_train_tensor.shape)

torch.Size([712, 13])
torch.Size([712, 1])


In [9]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

In [10]:
x_batch, y_batch = next(iter(train_loader))

print(x_batch.shape)
print(y_batch.shape)

torch.Size([64, 13])
torch.Size([64, 1])


In [53]:
torch.manual_seed(SEED)

model = TitanicMLP(input_dim=13)

loss_fn = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
EPOCHS = 100

best_val_accuracy = 0.0
best_epoch = 0

for epoch in range(EPOCHS):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0
    
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()
        
        logits = model(x_batch)
        loss = loss_fn(logits, y_batch)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        predictions = (torch.sigmoid(logits) >= 0.5).float()
        
        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)
    
    mean_train_loss = running_loss / len(train_loader)
    accuracy = correct / total if total > 0 else 0

    model.eval()
    
    val_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            logits = model(x_batch)
            loss = loss_fn(logits, y_batch)
            
            val_loss += loss.item()
            
            predictions = (torch.sigmoid(logits) >= 0.5).float()
            
            correct += (predictions == y_batch).sum().item()
            total += y_batch.size(0)
            
    mean_val_loss = val_loss / len(val_loader)
    val_accuracy = correct / total
    
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_epoch = epoch + 1
    
    print(f"Epoch {epoch+1}/{EPOCHS} - Train Loss: {mean_train_loss:.4f}, Train Acc: {accuracy:.4f}, Val Loss: {mean_val_loss:.4f}, Val Acc: {val_accuracy:.4f}")
    
print(
    f"Best Val Acc: {best_val_accuracy:.4f} "
    f"at epoch {best_epoch}"
)

In [43]:
class TitanicEmbeddingMLP(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.sex_embedding = nn.Embedding(2, 2)
        self.embarked_embedding = nn.Embedding(3, 2)
        self.pclass_embedding = nn.Embedding(3, 2)
        
        self.net = nn.Sequential(
            nn.Linear(11, 32, bias=False),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            
            nn.Linear(32, 32, bias=False),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            
            nn.Linear(32, 1)
        )
        
    def forward(self, x_num, sex, embarked, pclass):
        sex_embed = self.sex_embedding(sex)
        embarked_embed = self.embarked_embedding(embarked)
        pclass_embed = self.pclass_embedding(pclass)
        
        x = torch.cat([x_num, sex_embed, embarked_embed, pclass_embed], dim=1)
        return self.net(x)

In [14]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

fold_scores = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X, y), start=1):
    print(f'\nFold {fold}')
    
    # 1. Split
    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]
    X_val = X.iloc[val_idx]
    y_val = y.iloc[val_idx]
    
    # 2. Preprocessing
    preprocessor = build_preprocessor('standard')
    
    X_train_processed = preprocessor.fit_transform(X_train)
    X_val_processed = preprocessor.transform(X_val)
    
    # 3. Tensors
    X_train_tensor = torch.tensor(X_train_processed, dtype=torch.float32)
    X_val_tensor = torch.tensor(X_val_processed, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
    y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).unsqueeze(1)
    
    # 4. Dataloaders
    train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=32, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=32, shuffle=False)
    
    # 5. Model, Loss, Optimizer
    torch.manual_seed(SEED)
    
    model = TitanicMLP(input_dim=13)
    
    loss_fn =  nn.BCEWithLogitsLoss()
    
    EPOCHS = 100
    
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
    
    best_val_accuracy = 0.0
    
    for epoch in range(EPOCHS):
        model.train()
        
        for x_batch, y_batch in train_loader:
            optimizer.zero_grad()
            
            logits = model(x_batch)
            loss = loss_fn(logits, y_batch)
            
            loss.backward()
            optimizer.step()
            
        model.eval()
        
        correct = 0
        total = 0
        
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                logits = model(x_batch)
                
                predictions = (torch.sigmoid(logits) >= 0.5).float()
                
                correct += (predictions == y_batch).sum().item()
                total += y_batch.size(0)
                
        val_accuracy = correct / total
        
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
    
        
            
    print(f"Best Val Acc: {best_val_accuracy:.4f}")
    
    fold_scores.append(best_val_accuracy)
    
fold_scores = np.array(fold_scores)

print("\nDNN 5-fold CV")
print("Scores:", fold_scores)
print("Mean:", fold_scores.mean())
print("Std:", fold_scores.std())


Fold 1
Best Val Acc: 0.8603

Fold 2
Best Val Acc: 0.8596

Fold 3
Best Val Acc: 0.8090

Fold 4
Best Val Acc: 0.8539

Fold 5
Best Val Acc: 0.8371

DNN 5-fold CV
Scores: [0.8603352  0.85955056 0.80898876 0.85393258 0.83707865]
Mean: 0.8439771514656957
Std: 0.019395896013548186


In [33]:
X = train.drop(columns="Survived")
y = train["Survived"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=SEED,
)

print(X_train.shape, X_val.shape)
print(y_train.shape, y_val.shape)

(712, 11) (179, 11)
(712,) (179,)


In [34]:
feature_pipeline = Pipeline([
    ('age_imputer', AgeImputer()),
    ('embarked_imputer', EmbarkedImputer()),
    ('has_cabin', HasCabinTransformer()),
])

X_train_ready = feature_pipeline.fit_transform(X_train)
X_val_ready = feature_pipeline.transform(X_test)

sex_map = {
    'male': 0,
    'female': 1
}

embarked_map = {
    'S': 0,
    'C': 1,
    'Q': 2,
}

pclass_map = {
    1: 0,
    2: 1,
    3: 2,
}

sex_train = X_train_ready['Sex'].map(sex_map)
embarked_train = X_train_ready['Embarked'].map(embarked_map)
pclass_train = X_train_ready['Pclass'].map(pclass_map)

In [35]:
print(sex_train.unique())
print(embarked_train.unique())
print(pclass_train.unique())

[0 1]
[0 1 2]
[2 1 0]


In [36]:
numerical_features = [
    'Age',
    'Fare',
    'SibSp',
    'Parch',
    'HasCabin',
]

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_num_train = scaler.fit_transform(X_train_ready[numerical_features])
X_num_val = scaler.transform(X_val_ready[numerical_features])

X_num_train = torch.tensor(X_num_train, dtype=torch.float32)
X_num_val = torch.tensor(X_num_val, dtype=torch.float32)

In [37]:
sex_train = torch.tensor(X_train_ready["Sex"].map(sex_map).values, dtype=torch.long)
embarked_train = torch.tensor(X_train_ready["Embarked"].map(embarked_map).values, dtype=torch.long)
pclass_train = torch.tensor(X_train_ready["Pclass"].map(pclass_map).values, dtype=torch.long)

sex_val = torch.tensor(X_val_ready["Sex"].map(sex_map).values, dtype=torch.long)
embarked_val = torch.tensor(X_val_ready["Embarked"].map(embarked_map).values, dtype=torch.long)
pclass_val = torch.tensor(X_val_ready["Pclass"].map(pclass_map).values, dtype=torch.long)

In [38]:
y_train_tensor = torch.tensor(y_train.values,dtype=torch.float32).unsqueeze(1)
y_val_tensor = torch.tensor(y_val.values,dtype=torch.float32).unsqueeze(1)

In [39]:
train_dataset = TensorDataset(X_num_train, sex_train, embarked_train, pclass_train, y_train_tensor)
val_dataset = TensorDataset(X_num_val, sex_val, embarked_val, pclass_val, y_val_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [44]:
torch.manual_seed(SEED)

model = TitanicEmbeddingMLP()

x_num, sex, embarked, pclass, y = next(iter(train_loader))

logits = model(x_num, sex, embarked, pclass)

print("x_num:", x_num.shape)
print("logits:", logits.shape)
print("y:", y.shape)

x_num: torch.Size([32, 5])
logits: torch.Size([32, 1])
y: torch.Size([32, 1])


In [48]:
X = train.drop(columns="Survived")
y = train["Survived"]

print(X.shape)
print(y.shape)

(891, 11)
(891,)


In [50]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

fold_scores = []

numerical_features = [
    'Age',
    'Fare',
    'SibSp',
    'Parch',
    'HasCabin',
]

sex_map = {
    'male': 0,
    'female': 1,
}

embarked_map = {
    'S': 0,
    'C': 1,
    'Q': 2,
}

pclass_map = {
    1: 0,
    2: 1,
    3: 2,
}

for fold, (train_idx, val_idx) in enumerate(cv.split(X, y), start=1):
    print(f'\nFold {fold}')
    
    # 1. Split
    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]
    X_val = X.iloc[val_idx]
    y_val = y.iloc[val_idx]
    
    # 2. feature preprocessing
    feature_pipeline = Pipeline([
        ('age_imputer', AgeImputer()),
        ('embarked_imputer', EmbarkedImputer()),
        ('has_cabin', HasCabinTransformer()),
    ])
    
    X_train_ready = feature_pipeline.fit_transform(X_train)
    X_val_ready = feature_pipeline.transform(X_val)
    
    scaler = StandardScaler()
    
    X_num_train = scaler.fit_transform(X_train_ready[numerical_features])
    X_num_val = scaler.transform(X_val_ready[numerical_features])
    
    # 3. Tensors
    X_num_train = torch.tensor(X_num_train, dtype=torch.float32)
    X_num_val = torch.tensor(X_num_val, dtype=torch.float32)
    
    sex_train = torch.tensor(X_train_ready['Sex'].map(sex_map).values, dtype=torch.long)
    sex_val = torch.tensor(X_val_ready['Sex'].map(sex_map).values, dtype=torch.long)
    
    embarked_train = torch.tensor(X_train_ready['Embarked'].map(embarked_map).values, dtype=torch.long)
    embarked_val = torch.tensor(X_val_ready['Embarked'].map(embarked_map).values, dtype=torch.long)
    
    pclass_train = torch.tensor(X_train_ready['Pclass'].map(pclass_map).values, dtype=torch.long)
    pclass_val = torch.tensor(X_val_ready['Pclass'].map(pclass_map).values, dtype=torch.long)
    
    y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
    y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).unsqueeze(1)
    
    # 4. Dataloaders
    train_dataset = TensorDataset(X_num_train, sex_train, embarked_train, pclass_train, y_train_tensor)
    val_dataset = TensorDataset(X_num_val, sex_val, embarked_val, pclass_val, y_val_tensor)
    
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
    
    # 5. Model, Loss, Optimizer
    torch.manual_seed(SEED)
    
    model = TitanicEmbeddingMLP()
    
    loss_fn =  nn.BCEWithLogitsLoss()
    
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
    
    EPOCHS = 100
    best_val_accuracy = 0.0
    
    for epoch in range(EPOCHS):
        model.train()
        
        for x_num, sex, embarked, pclass, y_batch in train_loader:
            
            optimizer.zero_grad()
            
            logits = model(x_num, sex, embarked, pclass)
            loss = loss_fn(logits, y_batch)
            
            loss.backward()
            optimizer.step()
            
        model.eval()

        correct = 0
        total = 0
        
        with torch.no_grad():
            for x_num, sex, embarked, pclass, y_batch in val_loader:
                
                logits = model(x_num, sex, embarked, pclass)
                
                predictions = (torch.sigmoid(logits) >= 0.5).float()
                
                correct += (predictions == y_batch).sum().item()
                
                total += y_batch.size(0)
                
        val_accuracy = correct / total
        
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
    
        
            
    print(f"Best Val Acc: {best_val_accuracy:.4f}")
    
    fold_scores.append(best_val_accuracy)
    
fold_scores = np.array(fold_scores)

print("\nEmbedding DNN 5-fold CV")
print("Scores:", fold_scores)
print("Mean:", fold_scores.mean())
print("Std:", fold_scores.std())


Fold 1
Best Val Acc: 0.8659

Fold 2
Best Val Acc: 0.8483

Fold 3
Best Val Acc: 0.8202

Fold 4
Best Val Acc: 0.8652

Fold 5
Best Val Acc: 0.8315

Embedding DNN 5-fold CV
Scores: [0.86592179 0.84831461 0.82022472 0.86516854 0.83146067]
Mean: 0.846218065407068
Std: 0.01813938956284956
